<a href="https://colab.research.google.com/github/chrisokura/portfolio/blob/main/notebooks/nfl_4th_down_llm_evaluator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NFL 4th Down Decision Evaluator — LLM-as-a-Judge

**Author:** Chris Okura · [linkedin.com/in/chrisokura](https://linkedin.com/in/chrisokura)

---

## Overview

This project applies **LLM-as-a-Judge** to evaluate NFL 4th-down coaching decisions — mirroring the production AI evaluation framework I built at Meta, applied to a public dataset anyone can run.

### What it does
1. Loads NFL play-by-play data from the public `nfl_data_py` dataset (nflfastR)
2. Extracts 4th-down plays with full game context (score, field position, time, yards-to-go)
3. Asks an LLM to evaluate whether the coach's decision (go / punt / field goal) was optimal — providing structured reasoning and a quality rating
4. Validates LLM calibration by measuring whether quality ratings correlate with actual **Win Probability Added (WPA)** outcomes across hundreds of plays

### Why LLM-as-a-Judge?
The LLM is not predicting outcomes — it is **evaluating decision quality** given the information available before the play. This mirrors how evaluation frameworks work in production AI systems: the evaluator scores a decision based on context and reasoning, then calibration is validated against ground-truth outcomes.

With enough plays, execution noise averages out: decisions rated "optimal" should have higher average WPA than decisions rated "suboptimal" — even though individual play outcomes are noisy.

### Dataset
- **nfl_data_py** — Python wrapper for the nflfastR R package
- Play-by-play data: 2022–2023 NFL seasons
- ~5,000 4th-down decisions with WPA, win probability, and full game context

### LLM
- Uses **OpenAI GPT-4o-mini** by default (swap for any model)
- Structured JSON output for reliable parsing
- Chain-of-thought reasoning for interpretability

## 1. Setup

In [6]:
# Install dependencies (Colab already has pandas, numpy, matplotlib, seaborn)

!pip install -q nfl_data_py --no-deps
!pip install -q appdirs fastparquet openai tqdm
print('✓ All dependencies installed.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nfl-data-py 0.3.3 requires numpy<2.0,>=1.0, but you have numpy 2.0.2 which is incompatible.
nfl-data-py 0.3.3 requires pandas<2.0,>=1.0, but you have pandas 2.2.2 which is incompatible.
✓ All dependencies installed.


In [7]:
import os
import json
import time
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from tqdm.auto import tqdm
from openai import OpenAI

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

print('Libraries loaded.')

Libraries loaded.


In [8]:
# ─── API KEY ─────────────────────────────────────────────────────────────────
# This notebook uses YOUR OWN OpenAI API key — no key is stored in this file.
#
# To add your key in Colab:
#   1. Click the 🔑 icon in the left sidebar (Secrets)
#   2. Add a secret named:  OPENAI_API_KEY
#   3. Paste your key as the value
#   4. Re-run this cell
#
# Get a key at: https://platform.openai.com/api-keys
# Cost estimate: ~$0.05–0.15 for 100 plays with gpt-4o-mini

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    import os
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    raise ValueError("No API key found. Add OPENAI_API_KEY to Colab Secrets (🔑 icon).")

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = 'gpt-4o-mini'  # swap to 'gpt-4o' for higher quality (~10x more expensive)

print(f'✓ API key loaded. Using model: {MODEL}')

✓ API key loaded. Using model: gpt-4o-mini


## 2. Load & Prepare Data

In [9]:
import nfl_data_py as nfl

SEASONS = [2022, 2023]  # reduced to 2 seasons for faster loading (~1-2 min)
print(f'Loading play-by-play data for {SEASONS}...')
print('This may take 1-2 minutes — downloading from nflfastR...')

pbp_raw = nfl.import_pbp_data(years=SEASONS)

# Keep only the columns we need
COLS = [
    'game_id', 'play_id', 'season', 'week', 'posteam', 'defteam',
    'down', 'ydstogo', 'yardline_100', 'qtr', 'game_seconds_remaining',
    'score_differential', 'play_type', 'epa', 'wp', 'wpa',
    'fourth_down_converted', 'fourth_down_failed',
    'field_goal_result', 'punt_blocked', 'desc'
]
COLS = [c for c in COLS if c in pbp_raw.columns]  # only keep cols that exist
pbp_raw = pbp_raw[COLS]

print(f'✓ Loaded {len(pbp_raw):,} total plays across {len(SEASONS)} seasons.')

Loading play-by-play data for [2022, 2023]...
This may take 1-2 minutes — downloading from nflfastR...
2022 done.
2023 done.
Downcasting floats.
✓ Loaded 99,099 total plays across 2 seasons.


In [ ]:
# Visualize Dataset
pbp_raw.head(25)

In [ ]:
# Filter to 4th down plays with a clear decision
fourth_downs = pbp_raw[
    (pbp_raw['down'] == 4) &
    (pbp_raw['play_type'].isin(['run', 'pass', 'punt', 'field_goal'])) &
    (pbp_raw['wpa'].notna()) &
    (pbp_raw['wp'].notna())
].copy()

def classify_decision(play_type):
    if play_type in ['run', 'pass']:
        return 'go'
    elif play_type == 'punt':
        return 'punt'
    elif play_type == 'field_goal':
        return 'field_goal'
    return 'unknown'

fourth_downs['decision'] = fourth_downs['play_type'].apply(classify_decision)
fourth_downs = fourth_downs[fourth_downs['decision'] != 'unknown']

# ─── HISTORICAL WPA LOOKUP TABLE ─────────────────────────────────────────────
# Avg WPA per (situation bin × decision type) across all plays.
# Used to compare expected value of LLM recommendation vs coach decision
# without outcome leakage on individual plays.

fourth_downs['ydstogo_bin'] = pd.cut(
    fourth_downs['ydstogo'],
    bins=[0, 1, 3, 5, 100],
    labels=['1', '2-3', '4-5', '6+']
)
fourth_downs['yardline_bin'] = pd.cut(
    fourth_downs['yardline_100'],
    bins=[0, 20, 40, 60, 100],
    labels=['red_zone', 'fg_range', 'midfield', 'own_territory']
)

wpa_lookup = (
    fourth_downs
    .groupby(['ydstogo_bin', 'yardline_bin', 'decision'])['wpa']
    .mean()
    .reset_index()
    .rename(columns={'wpa': 'avg_wpa'})
)

print(f'4th down plays: {len(fourth_downs):,}')
print(f"\nDecision breakdown:\n{fourth_downs['decision'].value_counts()}")
print(f"\nHistorical avg WPA by situation + decision type (lookup table):")
print(wpa_lookup.sort_values(['ydstogo_bin', 'yardline_bin']).to_string(index=False))

In [ ]:
# Sample for LLM evaluation
# N_SAMPLE=30 for testing, increase to 300+ for meaningful calibration results
N_SAMPLE = 30

groups = []
for decision in ['go', 'punt', 'field_goal']:
    subset = fourth_downs[fourth_downs['decision'] == decision]
    n = min(len(subset), max(1, N_SAMPLE // 3))
    if n > 0:
        groups.append(subset.sample(n, random_state=42))

sampled = pd.concat(groups).sample(
    min(N_SAMPLE, sum(len(g) for g in groups)), random_state=42
).reset_index(drop=True)

print(f'Sample: {len(sampled)} plays')
print(sampled['decision'].value_counts())

In [ ]:
# Preview the sample
sampled[['game_id', 'posteam', 'defteam', 'qtr', 'ydstogo',
         'yardline_100', 'score_differential', 'wp', 'wpa', 'decision']].head(10)

## 3. LLM Evaluation

For each 4th down play, we:
1. Show the LLM the full game situation AND the coach's actual decision
2. Ask the LLM to evaluate whether that decision was optimal — with chain-of-thought reasoning
3. Parse the structured JSON response: quality rating, confidence, reasoning, key factors

The LLM is acting as a **judge**, not a predictor. It evaluates decision quality based on the context available before the play — just like a human analyst reviewing game film.

**Calibration hypothesis:** If the LLM is well-calibrated, decisions rated "optimal" should have higher average WPA than decisions rated "suboptimal" across a large enough sample — even though individual play outcomes include execution noise.

In [ ]:
# ─── COMPUTE BASE RATES FROM DATA ────────────────────────────────────────────
# These are used in the LLM system prompt so the model reasons from
# actual historical rates rather than estimated values.

# 4th down conversion rate by yards-to-go bin (go-for-it plays only)
go_plays = fourth_downs[fourth_downs['decision'] == 'go'].copy()
go_plays['converted'] = go_plays['fourth_down_converted'].fillna(0).astype(int)
conversion_rates = go_plays.groupby('ydstogo_bin', observed=True)['converted'].mean()

# Field goal success rate by distance bin
fg_plays = fourth_downs[fourth_downs['decision'] == 'field_goal'].copy()
fg_plays['fg_made'] = (fg_plays['field_goal_result'] == 'made').astype(int)
fg_plays['fg_distance'] = fg_plays['yardline_100'] + 17
fg_plays['fg_bin'] = pd.cut(
    fg_plays['fg_distance'],
    bins=[0, 37, 47, 57, 100],
    labels=['≤37yd', '38-47yd', '48-57yd', '58yd+']
)
fg_rates = fg_plays.groupby('fg_bin', observed=True)['fg_made'].mean()

# Net punt (avg change in field position)
punt_plays = fourth_downs[fourth_downs['decision'] == 'punt'].copy()
# yardline_100 before punt minus yardline_100 implied by WPA context
# Approximate net punt as the mean WPA-weighted position change isn't available,
# so use league average net punt from the data description field if available,
# otherwise fall back to known league average (~40 yards)
avg_net_punt = 40  # yards — well-established NFL average

print('4th down conversion rates (from your dataset):')
for bin_label, rate in conversion_rates.items():
    print(f'  {bin_label} yards: {rate:.1%}')

print('\nField goal success rates (from your dataset):')
for bin_label, rate in fg_rates.items():
    print(f'  {bin_label}: {rate:.1%}')

print(f'\nNet punt (league average): ~{avg_net_punt} yards')

In [ ]:
# Build system prompt dynamically from computed base rates
conv_lines = '\n'.join(
    f'  {b} yards: ~{r:.0%}' for b, r in conversion_rates.items()
)
fg_lines = '\n'.join(
    f'  {b}: ~{r:.0%}' for b, r in fg_rates.items()
)

SYSTEM_PROMPT = (
    "You are an expert NFL analytics coach evaluating 4th-down decisions.\n"
    "You will be shown a game situation and the decision the coach made.\n"
    "Your job is to evaluate whether that decision was optimal for maximizing win probability.\n\n"
    "Use these base rates computed from 2022-2023 NFL play-by-play data:\n\n"
    "4th down conversion rates by yards to go:\n"
    + conv_lines +
    "\n\nField goal success rates by distance:\n"
    + fg_lines +
    f"\n\nFailed conversion: opponent takes over at current spot (costly field position).\n"
    f"Net punt: ~{avg_net_punt} yards average field position change.\n\n"
    "Rate the decision as:\n"
    "  optimal    — clearly the highest expected value choice given the situation\n"
    "  acceptable — reasonable, defensible, though not necessarily best\n"
    "  suboptimal — a different decision would have meaningfully higher expected value\n\n"
    "Respond ONLY with valid JSON. No extra text."
)

RESPONSE_SCHEMA = {
    "decision_quality": "string — one of: optimal, acceptable, suboptimal",
    "confidence": "integer 1-5 (1=low, 5=high)",
    "reasoning": "string — 2-3 sentences explaining why this decision was or wasn't optimal",
    "key_factors": "list of 2-3 strings — the most important factors in this evaluation"
}

def build_prompt(row):
    mins = int(row['game_seconds_remaining'] // 60)
    secs = int(row['game_seconds_remaining'] % 60)
    score_str = (
        f"Leading by {abs(int(row['score_differential']))} points"
        if row['score_differential'] > 0
        else f"Trailing by {abs(int(row['score_differential']))} points"
        if row['score_differential'] < 0
        else "Tied"
    )
    yardline_str = (
        f"own {100 - int(row['yardline_100'])} yard line"
        if row['yardline_100'] > 50
        else f"opponent's {int(row['yardline_100'])} yard line"
    )
    fg_distance = int(row['yardline_100']) + 17

    return textwrap.dedent(f"""
        Game situation:
        - Quarter: {int(row['qtr'])} | Time remaining: {mins}:{secs:02d}
        - Field position: {yardline_str} ({int(row['yardline_100'])} yards from end zone)
        - 4th and {int(row['ydstogo'])} yards to go
        - Potential field goal distance: ~{fg_distance} yards
        - Score: {score_str}
        - Win probability: {row['wp']:.1%}
        - Offense: {row['posteam']} | Defense: {row['defteam']}

        Coach's decision: {row['decision'].upper().replace('_', ' ')}

        Evaluate this decision. Was it optimal given the situation?
        Respond in this JSON schema: {json.dumps(RESPONSE_SCHEMA, indent=2)}
    """).strip()

print('System prompt built from computed base rates:')
print(SYSTEM_PROMPT)

In [ ]:
def evaluate_play(row, retries=2):
    """Call LLM to evaluate the coach's decision. Returns quality rating + reasoning."""
    prompt = build_prompt(row)

    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.2,
                max_tokens=400,
            )
            result = json.loads(response.choices[0].message.content)
            # Attach ground truth for calibration analysis
            result['decision']     = row['decision']
            result['wpa']          = row['wpa']
            result['wp']           = row['wp']
            result['ydstogo']      = row['ydstogo']
            result['yardline_100'] = row['yardline_100']
            result['game_id']      = row['game_id']
            return result
        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                return {'error': str(e), 'decision': row['decision']}

print('Evaluator defined. Ready to run.')

In [ ]:
# ─── RUN EVALUATION ──────────────────────────────────────────────────────────
# Estimated cost: ~$0.05-0.15 for 100 plays with gpt-4o-mini
#                 ~$0.50-1.50 for 100 plays with gpt-4o

results = []
errors = 0

for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc='Evaluating plays'):
    result = evaluate_play(row)
    if 'error' in result:
        errors += 1
    results.append(result)
    time.sleep(0.3)  # rate limit buffer

results_df = pd.DataFrame([r for r in results if 'error' not in r])
print(f'\nEvaluated: {len(results_df)} plays | Errors: {errors}')
results_df.head(3)

Evaluating plays:   0%|          | 0/99 [00:00<?, ?it/s]

## 3b. LLM as Decision-Maker

In this section the LLM makes a **blind recommendation** — it sees the game situation but NOT the coach's actual decision.

We then look up the historical avg WPA for:
- The LLM's recommendation in that situation
- The coach's actual decision in that situation

And compare: did the LLM recommend a decision with higher expected value?

This is a fair comparison because we're using historical averages across thousands of plays — not individual outcome WPA — so execution noise is averaged out.

In [ ]:
DECISION_SYSTEM_PROMPT = (
    "You are an expert NFL analytics coach making 4th-down decisions.\n"
    "Given the game situation, recommend the optimal decision to maximize win probability.\n\n"
    "Use these base rates computed from 2022-2023 NFL play-by-play data:\n\n"
    "4th down conversion rates by yards to go:\n"
    + conv_lines +
    "\n\nField goal success rates by distance:\n"
    + fg_lines +
    f"\n\nFailed conversion: opponent takes over at current spot.\n"
    f"Net punt: ~{avg_net_punt} yards average field position change.\n\n"
    "Recommend one of: go, punt, field_goal\n\n"
    "Respond ONLY with valid JSON. No extra text."
)

DECISION_SCHEMA = {
    "recommendation": "string — one of: go, punt, field_goal",
    "confidence": "integer 1-5",
    "reasoning": "string — 2-3 sentences explaining the expected value calculation",
    "key_factors": "list of 2-3 strings"
}

def build_decision_prompt(row):
    """Same context as evaluator prompt but WITHOUT the coach's decision."""
    mins = int(row['game_seconds_remaining'] // 60)
    secs = int(row['game_seconds_remaining'] % 60)
    score_str = (
        f"Leading by {abs(int(row['score_differential']))} points"
        if row['score_differential'] > 0
        else f"Trailing by {abs(int(row['score_differential']))} points"
        if row['score_differential'] < 0
        else "Tied"
    )
    yardline_str = (
        f"own {100 - int(row['yardline_100'])} yard line"
        if row['yardline_100'] > 50
        else f"opponent's {int(row['yardline_100'])} yard line"
    )
    fg_distance = int(row['yardline_100']) + 17

    return textwrap.dedent(f"""
        Game situation:
        - Quarter: {int(row['qtr'])} | Time remaining: {mins}:{secs:02d}
        - Field position: {yardline_str} ({int(row['yardline_100'])} yards from end zone)
        - 4th and {int(row['ydstogo'])} yards to go
        - Potential field goal distance: ~{fg_distance} yards
        - Score: {score_str}
        - Win probability: {row['wp']:.1%}
        - Offense: {row['posteam']} | Defense: {row['defteam']}

        What should the offense do?
        Respond in this JSON schema: {json.dumps(DECISION_SCHEMA, indent=2)}
    """).strip()

def make_decision(row, retries=2):
    """LLM makes a blind recommendation without knowing the coach's choice."""
    prompt = build_decision_prompt(row)
    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": DECISION_SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.2,
                max_tokens=300,
            )
            result = json.loads(response.choices[0].message.content)
            result['actual_decision'] = row['decision']
            result['ydstogo_bin']     = str(row['ydstogo_bin'])
            result['yardline_bin']    = str(row['yardline_bin'])
            result['wpa']             = row['wpa']
            result['ydstogo']         = row['ydstogo']
            result['yardline_100']    = row['yardline_100']
            result['game_id']         = row['game_id']
            return result
        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                return {'error': str(e), 'actual_decision': row['decision']}

print('Decision-maker defined with data-driven base rates.')

In [ ]:
# ─── RUN DECISION-MAKER ──────────────────────────────────────────────────────
decisions_raw = []
errors = 0

for i, (_, row) in enumerate(tqdm(sampled.iterrows(), total=len(sampled), desc='LLM decisions')):
    result = make_decision(row)
    if 'error' in result:
        errors += 1
        if errors <= 3:
            print(f'Error on play {i}: {result["error"]}')
    decisions_raw.append(result)
    time.sleep(0.3)

decisions_df = pd.DataFrame([r for r in decisions_raw if 'error' not in r])
print(f'\nDecisions made: {len(decisions_df)} | Errors: {errors}')
print(f"\nLLM recommendation breakdown:\n{decisions_df['recommendation'].value_counts()}")
print(f"\nActual coach breakdown:\n{decisions_df['actual_decision'].value_counts()}")

In [ ]:
# ─── EXPECTED VALUE COMPARISON ───────────────────────────────────────────────
# Look up historical avg WPA for LLM's recommendation vs coach's actual decision
# in the same situation bin. Positive EV lift = LLM recommended higher-value decision.

decisions_df['recommendation'] = decisions_df['recommendation'].str.strip().str.lower()

# Join expected WPA for LLM recommendation
decisions_df = decisions_df.merge(
    wpa_lookup.rename(columns={'decision': 'recommendation', 'avg_wpa': 'llm_expected_wpa'}),
    on=['ydstogo_bin', 'yardline_bin', 'recommendation'],
    how='left'
)

# Join expected WPA for coach's actual decision
decisions_df = decisions_df.merge(
    wpa_lookup.rename(columns={'decision': 'actual_decision', 'avg_wpa': 'coach_expected_wpa'}),
    on=['ydstogo_bin', 'yardline_bin', 'actual_decision'],
    how='left'
)

decisions_df['ev_lift'] = decisions_df['llm_expected_wpa'] - decisions_df['coach_expected_wpa']
decisions_df['llm_matches_coach'] = decisions_df['recommendation'] == decisions_df['actual_decision']

avg_llm_ev    = decisions_df['llm_expected_wpa'].mean()
avg_coach_ev  = decisions_df['coach_expected_wpa'].mean()
avg_ev_lift   = decisions_df['ev_lift'].mean()
pct_higher_ev = (decisions_df['ev_lift'] > 0).mean()
pct_agree     = decisions_df['llm_matches_coach'].mean()

print('=' * 60)
print('LLM DECISION-MAKER: EXPECTED VALUE COMPARISON')
print('=' * 60)
print(f'Avg expected WPA — LLM recommendation : {avg_llm_ev:+.4f}')
print(f'Avg expected WPA — Coach decision      : {avg_coach_ev:+.4f}')
print(f'Avg EV lift (LLM − Coach)              : {avg_ev_lift:+.4f}')
print(f'% plays LLM recommended higher-EV decision: {pct_higher_ev:.1%}')
print(f'% plays LLM agreed with coach             : {pct_agree:.1%}')
print('=' * 60)
print()
print('Positive EV lift = LLM identified a historically higher expected value decision.')
print('This comparison uses historical avg WPA per decision type — not individual outcomes.')

decisions_df[['game_id', 'actual_decision', 'recommendation',
              'coach_expected_wpa', 'llm_expected_wpa', 'ev_lift',
              'confidence', 'reasoning']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('LLM Decision-Maker: Expected Value vs Coach', fontsize=14, fontweight='bold')

# ── Plot 1: Avg expected WPA — LLM vs Coach ───────────────────────────────────
ax = axes[0]
labels = ['LLM\nRecommendation', 'Coach\nDecision']
values = [avg_llm_ev, avg_coach_ev]
colors = ['#06b6d4', '#8b5cf6']
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Avg Expected WPA\nLLM vs Coach', fontweight='bold')
ax.set_ylabel('Avg Historical WPA for Decision Type')
for bar, val in zip(bars, values):
    offset = 0.0005 if val >= 0 else -0.003
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f'{val:+.4f}', ha='center', va='bottom', fontweight='bold')

# ── Plot 2: EV lift per play ──────────────────────────────────────────────────
ax = axes[1]
ev_clean = decisions_df['ev_lift'].dropna()
bar_colors = ['#10b981' if v > 0 else '#ef4444' for v in ev_clean]
ax.bar(range(len(ev_clean)), ev_clean.values, color=bar_colors, edgecolor='white')
ax.axhline(0, color='gray', linewidth=1, linestyle='--')
ax.set_title(f'EV Lift per Play\n(LLM − Coach expected WPA)', fontweight='bold')
ax.set_xlabel('Play')
ax.set_ylabel('EV Lift')
ax.text(0.98, 0.95, f'LLM higher EV: {pct_higher_ev:.0%}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, color='#10b981', fontweight='bold')

# ── Plot 3: LLM vs coach decision breakdown ───────────────────────────────────
ax = axes[2]
comparison = pd.DataFrame({
    'LLM': decisions_df['recommendation'].value_counts(),
    'Coach': decisions_df['actual_decision'].value_counts()
}).fillna(0)
comparison.plot(kind='bar', ax=ax, color=['#06b6d4', '#8b5cf6'],
                edgecolor='white', linewidth=1.2)
ax.set_title('Decision Distribution\nLLM vs Coach', fontweight='bold')
ax.set_xlabel('Decision Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
ax.legend()

plt.tight_layout()
plt.savefig('4th_down_decisions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as 4th_down_decisions.png')

## 4. Analysis & Results

In [ ]:
# ─── CALIBRATION ANALYSIS ────────────────────────────────────────────────────
# Core question: do LLM quality ratings correlate with actual WPA outcomes?
# Well-calibrated evaluator: optimal → higher avg WPA, suboptimal → lower avg WPA

results_df['wpa'] = pd.to_numeric(results_df['wpa'], errors='coerce')
results_df['confidence'] = pd.to_numeric(results_df['confidence'], errors='coerce')

# Avg WPA and % positive WPA by quality rating
calibration = results_df.groupby('decision_quality')['wpa'].agg(
    avg_wpa='mean',
    median_wpa='median',
    pct_positive=lambda x: (x > 0).mean(),
    count='count'
).reindex(['optimal', 'acceptable', 'suboptimal']).dropna()

# Quality distribution
quality_counts = results_df['decision_quality'].value_counts()
pct_optimal    = quality_counts.get('optimal', 0) / len(results_df)
pct_suboptimal = quality_counts.get('suboptimal', 0) / len(results_df)

print('=' * 60)
print('LLM QUALITY RATING DISTRIBUTION')
print('=' * 60)
for q in ['optimal', 'acceptable', 'suboptimal']:
    n = quality_counts.get(q, 0)
    print(f'  {q:12s}: {n:3d} plays ({n/len(results_df):.1%})')

print()
print('CALIBRATION: Avg WPA by LLM Quality Rating')
print('(Well-calibrated = optimal > acceptable > suboptimal)')
print('=' * 60)
print(calibration[['avg_wpa', 'median_wpa', 'pct_positive', 'count']].round(4).to_string())
print('=' * 60)

# Is the ordering correct?
if len(calibration) >= 2:
    ratings = calibration['avg_wpa']
    if 'optimal' in ratings.index and 'suboptimal' in ratings.index:
        lift = ratings.get('optimal', 0) - ratings.get('suboptimal', 0)
        print(f'\nCalibration lift (optimal − suboptimal avg WPA): {lift:+.4f}')
        print('Positive lift = LLM ratings align with win probability outcomes')

In [ ]:
# Decision quality distribution
quality_counts = results_df['decision_quality'].value_counts()
print('LLM decision quality ratings:')
for q, n in quality_counts.items():
    pct = n / len(results_df)
    print(f'  {q:12s}: {n:3d} ({pct:.1%})')

In [ ]:
# Average confidence by scenario
results_df['confidence'] = pd.to_numeric(results_df['confidence'], errors='coerce')

conf_by_decision = results_df.groupby('actual_decision')['confidence'].mean()
print('Average LLM confidence by actual coach decision:')
print(conf_by_decision.round(2))

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NFL 4th Down — LLM-as-a-Judge Calibration Results', fontsize=14, fontweight='bold')

# ── Plot 1: Avg WPA by quality rating ────────────────────────────────────────
ax = axes[0]
q_order = [q for q in ['optimal', 'acceptable', 'suboptimal'] if q in calibration.index]
wpa_vals = [calibration.loc[q, 'avg_wpa'] for q in q_order]
colors = {'optimal': '#10b981', 'acceptable': '#f59e0b', 'suboptimal': '#ef4444'}
bar_colors = [colors[q] for q in q_order]
bars = ax.bar(q_order, wpa_vals, color=bar_colors, edgecolor='white', linewidth=1.5)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Avg WPA by LLM Quality Rating\n(calibration check)', fontweight='bold')
ax.set_ylabel('Avg Win Probability Added')
ax.set_xlabel('LLM Rating')
for bar, val in zip(bars, wpa_vals):
    offset = 0.001 if val >= 0 else -0.003
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f'{val:+.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# ── Plot 2: % Positive WPA by quality rating ─────────────────────────────────
ax = axes[1]
pct_vals = [calibration.loc[q, 'pct_positive'] for q in q_order]
bars = ax.bar(q_order, pct_vals, color=bar_colors, edgecolor='white', linewidth=1.5)
ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', label='50% baseline')
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('% Positive WPA Outcomes\nby LLM Quality Rating', fontweight='bold')
ax.set_ylabel('% of Plays with Positive WPA')
ax.set_xlabel('LLM Rating')
ax.legend(fontsize=9)
for bar, val in zip(bars, pct_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.0%}', ha='center', va='bottom', fontweight='bold')

# ── Plot 3: Quality rating distribution by decision type ─────────────────────
ax = axes[2]
cross = pd.crosstab(results_df['decision'], results_df['decision_quality'])
cross = cross.reindex(columns=['optimal', 'acceptable', 'suboptimal'], fill_value=0)
cross.plot(kind='bar', ax=ax,
           color=[colors['optimal'], colors['acceptable'], colors['suboptimal']],
           edgecolor='white', linewidth=1.2)
ax.set_title('LLM Quality Ratings\nby Coach Decision Type', fontweight='bold')
ax.set_xlabel('Coach Decision')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='LLM Rating', fontsize=9)

plt.tight_layout()
plt.savefig('4th_down_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as 4th_down_calibration.png')

In [ ]:
# ── Disagreement deep-dive: where LLM and coach differ most ──────────────────

disagree = results_df[~results_df['llm_matches_actual']].copy()
disagree_go = disagree[
    (disagree['actual_decision'] != 'go') &
    (disagree['recommendation'] == 'go')
]

print(f'Total disagreements: {len(disagree)} ({len(disagree)/len(results_df):.1%} of plays)')
print(f'LLM says GO, coach did not: {len(disagree_go)} ({len(disagree_go)/len(results_df):.1%})')
print()
print('Disagreement breakdown (LLM recommendation vs actual):')
print(pd.crosstab(disagree['recommendation'], disagree['actual_decision'], margins=True))

In [ ]:
# ── Sample LLM reasoning for interesting plays ────────────────────────────────

print('=' * 70)
print('SAMPLE LLM EVALUATIONS — Where LLM and Coach Disagreed')
print('=' * 70)

show = disagree_go.head(3) if len(disagree_go) >= 3 else disagree.head(3)

for i, (_, row) in enumerate(show.iterrows(), 1):
    print(f'\n[{i}] {row["game_id"]} | 4th & {int(row["ydstogo"])} | '
          f'{int(row["yardline_100"])} yds from end zone')
    print(f'    Coach: {row["actual_decision"].upper()} | '
          f'LLM: {row["recommendation"].upper()} (confidence: {row["confidence"]}/5)')
    print(f'    EPA-optimal: {row["epa_optimal"].upper()}')
    print(f'    Reasoning: {row["reasoning"]}')
    if isinstance(row.get('key_factors'), list):
        print(f'    Key factors: {" | ".join(row["key_factors"])}')
    print()

## 6. Calibration Analysis

A well-calibrated evaluator should have higher confidence on plays where it agrees with the benchmark, and lower confidence on ambiguous plays. This is the same calibration analysis I applied to the human-in-the-loop components of the Meta evaluation framework.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('LLM Calibration Analysis', fontsize=13, fontweight='bold')

# ── Confidence vs EPA agreement ───────────────────────────────────────────────
ax = axes[0]
for matches, label, color in [
    (True, 'Agrees with EPA-optimal', '#10b981'),
    (False, 'Disagrees with EPA-optimal', '#ef4444')
]:
    subset = results_df[results_df['llm_matches_epa'] == matches]
    if len(subset) > 0:
        ax.hist(subset['confidence'], bins=5, range=(0.5, 5.5),
                label=f'{label} (n={len(subset)})',
                color=color, alpha=0.6, edgecolor='white')
ax.set_xlabel('LLM Confidence Score')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution by EPA Agreement')
ax.legend(fontsize=9)

# ── Confidence vs ydstogo (uncertainty should increase with ambiguity) ─────────
ax = axes[1]
ydstogo_bins = pd.cut(results_df['ydstogo'], bins=[0, 1, 3, 5, 10, 20],
                       labels=['1', '2-3', '4-5', '6-10', '10+'])
conf_by_yds = results_df.groupby(ydstogo_bins)['confidence'].mean()
ax.bar(conf_by_yds.index.astype(str), conf_by_yds.values,
       color='#06b6d4', edgecolor='white', linewidth=1.5)
ax.set_xlabel('Yards to Go')
ax.set_ylabel('Avg LLM Confidence')
ax.set_ylim(1, 5)
ax.set_title('Confidence vs Yards to Go\n(1-2 yards = easier decision)')

plt.tight_layout()
plt.savefig('4th_down_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

### What we measured
This notebook validates whether an LLM can serve as a calibrated judge of NFL 4th-down decision quality — without knowing the outcome in advance.

**Calibration hypothesis:** If the LLM's quality ratings are meaningful, decisions rated "optimal" should produce higher average WPA than decisions rated "suboptimal" across a large sample. Execution noise on individual plays averages out.

### Key metrics
| Metric | Value |
|--------|-------|
| % decisions rated optimal | ~X% |
| % decisions rated suboptimal | ~X% |
| Avg WPA — optimal decisions | ~+X.XXXX |
| Avg WPA — suboptimal decisions | ~-X.XXXX |
| Calibration lift (optimal − suboptimal) | ~+X.XXXX |

*(Run with N=300+ for stable estimates)*

### Connection to production evaluation work
This directly mirrors the LLM-as-a-Judge framework I built at Meta:
- **Structured rubric** with explicit quality dimensions (optimal / acceptable / suboptimal)
- **Chain-of-thought reasoning** for interpretability and auditability
- **Calibration analysis** to validate the evaluator against ground-truth outcomes
- **JSON-structured output** for downstream aggregation and analysis
- **Separation of decision quality from execution** — the same challenge as separating model quality from data noise in production AI systems

### Extensions
- Run with N=500+ plays for statistically robust calibration curves
- Compare evaluator quality across models: GPT-4o vs Claude 3.5 vs Llama 3
- Add confidence calibration: do high-confidence ratings have larger WPA separation?
- Apply the same framework to other sports or business decision domains

In [ ]:
# Save results
results_df.to_csv('nfl_4th_down_llm_results.csv', index=False)
print('Results saved to nfl_4th_down_llm_results.csv')
print(f'\nFinal summary:')
print(f'  Plays evaluated : {len(results_df)}')
print(f'  LLM vs EPA agreement: {llm_vs_epa:.1%}')
print(f'  Coach vs EPA agreement: {actual_vs_epa:.1%}')